# Universidad San Ignacio de Loyola (USIL) — Facultad de Ingeniería
## Notebook 02: Auditoría de Calidad (2015–2025) y Validación Cruzada CSV vs. DAX (`Dashboard_V2.pbix`)

**Artículo:** *La generación que espera: análisis visual del desempleo juvenil, la exclusión NEET por género y territorio, la informalidad y la pobreza laboral en América Latina* (Revista objetivo: **IJSDP – IIETA, Scopus Q3**)

**Responsabilidad Única (Principio SRP):** Operar en **modo solo lectura** sobre `data/processed/ilostat_latam_2010_2026_master_long.csv` (sin modificar ningún archivo de `data/processed/`), replicar exactamente el filtrado de Power Query (`2015–2025`, 20 países soberanos sin agregado `X21`) y auditar matemáticamente los **14 controles numéricos** reportados en `powerbi/Dashboard_V2.pbix`, en el artículo final (`01_Articulo_ES_Grupo01.docx` / `02_Articulo_EN_Grupo01.docx`) y en el Anexo Técnico (`03_Anexo_Tecnico_Grupo01.docx`).

In [1]:
import os
import pandas as pd

csv_path = os.path.join('..', 'data', 'processed', 'ilostat_latam_2010_2026_master_long.csv')
df_master = pd.read_csv(csv_path)

# Filtro oficial del estudio (idéntico al aplicado en Power Query de Dashboard_V2.pbix)
df = df_master[
    (df_master['Year'] >= 2015) &
    (df_master['Year'] <= 2025) &
    (~df_master['Is_Regional_Aggregate'].astype(bool))
].copy()

key_cols = ['ISO3', 'Year', 'Sex', 'Age_Group', 'Area_Type', 'Indicator_Short']
dups = int(df.duplicated(subset=key_cols).sum())
out_range = int(((df['Value_Pct'] < 0) | (df['Value_Pct'] > 100)).sum())

print(f'Registros en archivo maestro (2010–2026): {len(df_master):,}')
print(f'Registros en subconjunto analítico (2015–2025, 20 países sin X21): {len(df):,}')
print(f'Países únicos (ISO3): {df["ISO3"].nunique()}')
print(f'Claves primarias duplicadas: {dups} | Valores fuera de [0, 100]: {out_range}')
assert len(df_master) == 23891 and len(df) == 15305 and df['ISO3'].nunique() == 20
assert dups == 0 and out_range == 0

Registros en archivo maestro (2010–2026): 23,891
Registros en subconjunto analítico (2015–2025, 20 países sin X21): 15,305
Países únicos (ISO3): 20
Claves primarias duplicadas: 0 | Valores fuera de [0, 100]: 0


---
### 1. Validación Dashboard 1 (`01 Desempleo`) — Evolución del Desempleo Juvenil vs. Adulto

In [2]:
ur = df[(df['Indicator_Short'] == 'UR_Modelled') & (df['Sex'] == 'Total')]
d1_rows = []
for yr in [2019, 2020, 2024, 2025]:
    y_mean = ur[(ur['Age_Group'] == 'Youth (15-24)') & (ur['Year'] == yr)]['Value_Pct'].mean()
    a_mean = ur[(ur['Age_Group'] == 'Adults (25+)') & (ur['Year'] == yr)]['Value_Pct'].mean()
    d1_rows.append({'Año': yr, 'Media Juvenil (%)': round(y_mean, 2), 'Media Adulta (%)': round(a_mean, 2), 'Ratio (R)': round(y_mean / a_mean, 2)})
df_d1 = pd.DataFrame(d1_rows)
delta_2020 = round(df_d1.loc[df_d1['Año']==2020, 'Media Juvenil (%)'].values[0] - df_d1.loc[df_d1['Año']==2019, 'Media Juvenil (%)'].values[0], 2)
delta_2025 = round(df_d1.loc[df_d1['Año']==2025, 'Media Juvenil (%)'].values[0] - df_d1.loc[df_d1['Año']==2024, 'Media Juvenil (%)'].values[0], 2)
print(f'Salto juvenil 2019 -> 2020: +{delta_2020:.2f} pp | Variación 2024 -> 2025: {delta_2025:.2f} pp')
display(df_d1)

Salto juvenil 2019 -> 2020: +4.24 pp | Variación 2024 -> 2025: -0.31 pp


,Año,Media Juvenil (%),Media Adulta (%),Ratio (R)
0,2019,15.84,4.81,3.29
1,2020,20.08,7.12,2.82
2,2024,13.94,4.58,3.04
3,2025,13.63,4.42,3.08


---
### 2. Validación Dashboard 2 (`02 Género NiNi`) — Brechas de Género en la Población NEET

In [3]:
neet = df[df['Indicator_Short'] == 'NEET_Youth_Modelled']
f15 = neet[(neet['Sex'] == 'Female') & (neet['Year'] == 2015)]['Value_Pct'].mean()
m15 = neet[(neet['Sex'] == 'Male') & (neet['Year'] == 2015)]['Value_Pct'].mean()
f25 = neet[(neet['Sex'] == 'Female') & (neet['Year'] == 2025)]['Value_Pct'].mean()
m25 = neet[(neet['Sex'] == 'Male') & (neet['Year'] == 2025)]['Value_Pct'].mean()
per_f25 = neet[(neet['Sex'] == 'Female') & (neet['Year'] == 2025) & (neet['ISO3'] == 'PER')]['Value_Pct'].iloc[0]
per_m25 = neet[(neet['Sex'] == 'Male') & (neet['Year'] == 2025) & (neet['ISO3'] == 'PER')]['Value_Pct'].iloc[0]
gaps25 = (neet[(neet['Sex']=='Female') & (neet['Year']==2025)].set_index('ISO3')['Value_Pct'] - neet[(neet['Sex']=='Male') & (neet['Year']==2025)].set_index('ISO3')['Value_Pct']).sort_values(ascending=False)

print(f'2015 (20 países): Mujeres={f15:.2f}%, Hombres={m15:.2f}%, Brecha={f15-m15:.2f} pp')
print(f'2025 (20 países): Mujeres={f25:.2f}%, Hombres={m25:.2f}%, Brecha={f25-m25:.2f} pp (cambio vs 2015: {(f25-m25)-(f15-m15):.2f} pp)')
print(f'Perú 2025 (país focal): Mujeres={per_f25:.2f}%, Hombres={per_m25:.2f}%, Brecha={per_f25-per_m25:.2f} pp')
print(f'Mayor brecha nacional 2025: {gaps25.index[0]} = {gaps25.iloc[0]:.2f} pp')

2015 (20 países): Mujeres=29.23%, Hombres=12.47%, Brecha=16.77 pp
2025 (20 países): Mujeres=26.02%, Hombres=12.83%, Brecha=13.19 pp (cambio vs 2015: -3.58 pp)
Perú 2025 (país focal): Mujeres=26.85%, Hombres=21.16%, Brecha=5.69 pp
Mayor brecha nacional 2025: GTM = 32.58 pp


---
### 3. Validación Dashboard 3 (`03 Territorio`) — Dispersión Nacional y Brecha Urbano-Rural NEET

In [4]:
n25 = neet[(neet['Sex'] == 'Total') & (neet['Year'] == 2025)].set_index('ISO3')['Value_Pct'].sort_values(ascending=False)
ur25 = df[(df['Indicator_Short'] == 'NEET_Urban_Rural') & (df['Sex'] == 'Total') & (df['Year'] == 2025)].pivot(
    index='ISO3', columns='Area_Type', values='Value_Pct'
).dropna(subset=['Urban', 'Rural'])
per_ur = df[(df['Indicator_Short'] == 'NEET_Urban_Rural') & (df['Sex'] == 'Total') & (df['ISO3'] == 'PER')].pivot(
    index='Year', columns='Area_Type', values='Value_Pct'
).dropna(subset=['Urban', 'Rural']).tail(1)

print(f'Nacional 2025 (n=20): Máx={n25.index[0]} ({n25.iloc[0]:.2f}%), Mín={n25.index[-1]} ({n25.iloc[-1]:.2f}%), Rango={n25.iloc[0]-n25.iloc[-1]:.2f} pp, Perú={n25["PER"]:.2f}%')
print(f'Pares Urbano/Rural 2025: n={len(ur25)}, Rural>Urbano={(ur25["Rural"]>ur25["Urban"]).sum()}/{len(ur25)}, Excepción={list(ur25[ur25["Rural"]<=ur25["Urban"]].index)}, Brecha media R-U={(ur25["Rural"]-ur25["Urban"]).mean():.2f} pp')
print('Último par Urbano/Rural disponible para Perú:')
display(per_ur)

Nacional 2025 (n=20): Máx=HND (30.81%), Mín=BOL (7.75%), Rango=23.06 pp, Perú=23.97%
Pares Urbano/Rural 2025: n=13, Rural>Urbano=12/13, Excepción=['ECU'], Brecha media R-U=5.38 pp
Último par Urbano/Rural disponible para Perú:


Area_Type,National,Rural,Urban
Year,,,
2022,22.78,12.55,25.1


---
### 4. Validación Dashboard 4 (`04 Vulnerabilidad`) — Informalidad (2025) y Pobreza Laboral (2023)

In [5]:
inf25 = df[(df['Indicator_Short'] == 'Informality_National') & (df['Sex'] == 'Total') & (df['Year'] == 2025)].pivot(
    index='ISO3', columns='Age_Group', values='Value_Pct'
).dropna(subset=['Youth (15-24)', 'Adults (25+)'])
wp23 = df[(df['Indicator_Short'] == 'Working_Poverty_Modelled') & (df['Sex'] == 'Total') & (df['Year'] == 2023)].pivot(
    index='ISO3', columns='Age_Group', values='Value_Pct'
).dropna(subset=['Youth (15-24)', 'Adults (25+)'])
inf23_y = df[(df['Indicator_Short'] == 'Informality_National') & (df['Sex'] == 'Total') & (df['Year'] == 2023) & (df['Age_Group'] == 'Youth (15-24)')].set_index('ISO3')['Value_Pct']
scat23 = pd.concat([inf23_y.rename('Informalidad_2023'), wp23['Youth (15-24)'].rename('Pobreza_2023')], axis=1).dropna()

print(f'Informalidad 2025 (n={len(inf25)} pares): Juvenil={inf25["Youth (15-24)"].mean():.2f}%, Adulta={inf25["Adults (25+)"].mean():.2f}%, Brecha={(inf25["Youth (15-24)"]-inf25["Adults (25+)"]).mean():.2f} pp')
print(f'Perú Informalidad 2025: Juvenil={inf25.loc["PER","Youth (15-24)"]:.2f}%, Adulta={inf25.loc["PER","Adults (25+)"]:.2f}%, Brecha={inf25.loc["PER","Youth (15-24)"]-inf25.loc["PER","Adults (25+)"]:.2f} pp')
print(f'Pobreza laboral 2023 (n={len(wp23)} pares): Juvenil={wp23["Youth (15-24)"].mean():.2f}%, Adulta={wp23["Adults (25+)"].mean():.2f}% | Perú: J={wp23.loc["PER","Youth (15-24)"]:.2f}%, A={wp23.loc["PER","Adults (25+)"]:.2f}%')
print(f'Dispersión juvenil 2023 (n={len(scat23)} pares): Punto Perú -> Inf={scat23.loc["PER","Informalidad_2023"]:.2f}%, Pob={scat23.loc["PER","Pobreza_2023"]:.2f}%')

Informalidad 2025 (n=15 pares): Juvenil=63.89%, Adulta=54.06%, Brecha=9.83 pp
Perú Informalidad 2025: Juvenil=85.68%, Adulta=67.95%, Brecha=17.73 pp
Pobreza laboral 2023 (n=19 pares): Juvenil=6.91%, Adulta=6.01% | Perú: J=8.04%, A=6.68%
Dispersión juvenil 2023 (n=15 pares): Punto Perú -> Inf=89.36%, Pob=8.04%


---
### 5. Tabla Resumen de Auditoría Automatizada (14 Controles: CSV vs. DAX en `Dashboard_V2.pbix`)

In [6]:
checks = [
    ('01 Desempleo', 'Desempleo juvenil 2019 (n=20)', round(df_d1.loc[df_d1['Año']==2019, 'Media Juvenil (%)'].iloc[0], 2), 15.84),
    ('01 Desempleo', 'Desempleo juvenil 2020 (n=20)', round(df_d1.loc[df_d1['Año']==2020, 'Media Juvenil (%)'].iloc[0], 2), 20.08),
    ('01 Desempleo', 'Desempleo juvenil 2025 (n=20)', round(df_d1.loc[df_d1['Año']==2025, 'Media Juvenil (%)'].iloc[0], 2), 13.63),
    ('01 Desempleo', 'Desempleo adulto 2025 (n=20)', round(df_d1.loc[df_d1['Año']==2025, 'Media Adulta (%)'].iloc[0], 2), 4.42),
    ('01 Desempleo', 'Ratio juvenil/adulto 2025', round(df_d1.loc[df_d1['Año']==2025, 'Ratio (R)'].iloc[0], 2), 3.08),
    ('02 Género NiNi', 'Brecha NEET F-M 2015 (pp)', round(f15 - m15, 2), 16.77),
    ('02 Género NiNi', 'Brecha NEET F-M 2025 (pp)', round(f25 - m25, 2), 13.19),
    ('02 Género NiNi', 'Brecha NEET Perú 2025 (pp)', round(per_f25 - per_m25, 2), 5.69),
    ('02 Género NiNi', 'Máxima brecha nacional 2025 (GTM)', round(gaps25.iloc[0], 2), 32.58),
    ('03 Territorio', 'Rango nacional NEET 2025 (HND-BOL)', round(n25.iloc[0] - n25.iloc[-1], 2), 23.06),
    ('03 Territorio', 'Brecha media Rural-Urbano 2025 (n=13)', round((ur25['Rural'] - ur25['Urban']).mean(), 2), 5.38),
    ('04 Vulnerabilidad', 'Informalidad juvenil 2025 (n=15)', round(inf25['Youth (15-24)'].mean(), 2), 63.89),
    ('04 Vulnerabilidad', 'Pobreza laboral juvenil 2023 (n=19)', round(wp23['Youth (15-24)'].mean(), 2), 6.91),
    ('04 Vulnerabilidad', 'Perú Dispersión 2023 (Inf. juvenil)', round(scat23.loc['PER', 'Informalidad_2023'], 2), 89.36),
]
audit_df = pd.DataFrame(checks, columns=['Dashboard', 'Control Verificado', 'Valor CSV (Python)', 'Valor DAX (PBIX)'])
audit_df['Error Absoluto'] = (audit_df['Valor CSV (Python)'] - audit_df['Valor DAX (PBIX)']).abs().round(2)
audit_df['Estado'] = audit_df['Error Absoluto'].apply(lambda e: 'OK (0.00)' if e == 0 else 'REVISAR')
assert (audit_df['Error Absoluto'] == 0).all(), 'Discrepancia detectada en auditoría'
display(audit_df)
print('AUDITORÍA COMPLETADA: 14/14 controles verificados con error absoluto = 0.00.')

,Dashboard,Control Verificado,Valor CSV (Python),Valor DAX (PBIX),Error Absoluto,Estado
0,01 Desempleo,Desempleo juvenil 2019 (n=20),15.84,15.84,0.0,OK (0.00)
1,01 Desempleo,Desempleo juvenil 2020 (n=20),20.08,20.08,0.0,OK (0.00)
2,01 Desempleo,Desempleo juvenil 2025 (n=20),13.63,13.63,0.0,OK (0.00)
3,01 Desempleo,Desempleo adulto 2025 (n=20),4.42,4.42,0.0,OK (0.00)
4,01 Desempleo,Ratio juvenil/adulto 2025,3.08,3.08,0.0,OK (0.00)
5,02 Género NiNi,Brecha NEET F-M 2015 (pp),16.77,16.77,0.0,OK (0.00)
6,02 Género NiNi,Brecha NEET F-M 2025 (pp),13.19,13.19,0.0,OK (0.00)
7,02 Género NiNi,Brecha NEET Perú 2025 (pp),5.69,5.69,0.0,OK (0.00)
8,02 Género NiNi,Máxima brecha nacional 2025 (GTM),32.58,32.58,0.0,OK (0.00)
9,03 Territorio,Rango nacional NEET 2025 (HND-BOL),23.06,23.06,0.0,OK (0.00)


AUDITORÍA COMPLETADA: 14/14 controles verificados con error absoluto = 0.00.
